# Simple JDBC Connection Test

A minimal notebook to verify the Neo4j JDBC driver and Unity Catalog JDBC connection are working.

**Before running this notebook**, create a test node in your Neo4j database:

```cypher
MERGE (n:JdbcTest {id: 1})
SET n.message = 'hello from Neo4j',
    n.created = datetime()
RETURN n
```

**Prerequisites**:
- Upload the Neo4j Unity Catalog Connector JAR to a UC Volume
- Install the connector JAR as a cluster library
- Update the configuration values in the next cell

In [ ]:
# =============================================================================
# CONFIGURATION - Update these values for your environment
# =============================================================================
NEO4J_HOST = "<your-neo4j-host>"           # e.g. "abc123.databases.neo4j.io"
NEO4J_USER = "<your-neo4j-user>"           # e.g. "neo4j"
NEO4J_PASSWORD = "<your-neo4j-password>"
NEO4J_DATABASE = "neo4j"                   # default database

# Path to the connector JAR in a UC Volume
JDBC_JAR_PATH = "/Volumes/<catalog>/<schema>/<volume>/neo4j-unity-catalog-connector.jar"

# Name for the Unity Catalog connection
UC_CONNECTION_NAME = "<catalog>.<schema>.neo4j_jdbc"

# Derived values (no need to edit)
JAVA_DEPENDENCIES = f'["{JDBC_JAR_PATH}"]'
NEO4J_JDBC_URL = f"jdbc:neo4j+s://{NEO4J_HOST}:7687/{NEO4J_DATABASE}"
NEO4J_JDBC_URL_SQL = f"{NEO4J_JDBC_URL}?enableSQLTranslation=true"

print(f"Neo4j Host: {NEO4J_HOST}")
print(f"JDBC URL: {NEO4J_JDBC_URL_SQL}")
print(f"Connection Name: {UC_CONNECTION_NAME}")
print(f"JDBC JAR Path: {JDBC_JAR_PATH}")

---

## Section 1: Verify JAR on Volume

In [ ]:
# Verify the connector JAR exists in the UC Volume
try:
    jar_dir = JDBC_JAR_PATH.rsplit('/', 1)[0]
    jar_name = JDBC_JAR_PATH.split('/')[-1]
    files = dbutils.fs.ls(jar_dir)
    found = any(f.name == jar_name for f in files)
    if found:
        print(f"[PASS] JAR found: {JDBC_JAR_PATH}")
    else:
        print(f"[FAIL] JAR not found: {jar_name}")
        print(f"  Files in {jar_dir}:")
        for f in files:
            print(f"    {f.name}")
except Exception as e:
    print(f"[FAIL] Cannot access volume: {e}")

---

## Section 2: Direct JDBC Tests

These tests use the Neo4j JDBC driver directly with Spark, without Unity Catalog's SafeSpark wrapper. Requires the connector JAR installed as a cluster library.

**Note**: `customSchema` is required because Neo4j's JDBC driver returns `NullType` during Spark's schema inference.

In [ ]:
# Direct JDBC - Basic connectivity test
import time

print("TEST: Direct JDBC - SELECT 1")
print(f"JDBC URL: {NEO4J_JDBC_URL_SQL}")

try:
    start_time = time.time()
    df = spark.read.format("jdbc") \
        .option("url", NEO4J_JDBC_URL_SQL) \
        .option("driver", "org.neo4j.jdbc.Neo4jDriver") \
        .option("user", NEO4J_USER) \
        .option("password", NEO4J_PASSWORD) \
        .option("query", "SELECT 1 AS value") \
        .option("customSchema", "value INT") \
        .load()

    results = df.collect()
    elapsed = (time.time() - start_time) * 1000

    print(f"\n[PASS] Connected in {elapsed:.1f}ms")
    df.show()
except Exception as e:
    print(f"\n[FAIL] {e}")

In [ ]:
# Direct JDBC - Read the JdbcTest node via SQL (translated to Cypher)
# Reads the node created by: MERGE (n:JdbcTest {id: 1}) SET n.message = 'hello from Neo4j'
import time

print("TEST: Direct JDBC - Read JdbcTest node")
print("SQL: SELECT id, message FROM JdbcTest WHERE id = 1")

try:
    start_time = time.time()
    df = spark.read.format("jdbc") \
        .option("url", NEO4J_JDBC_URL_SQL) \
        .option("driver", "org.neo4j.jdbc.Neo4jDriver") \
        .option("user", NEO4J_USER) \
        .option("password", NEO4J_PASSWORD) \
        .option("query", "SELECT id, message FROM JdbcTest WHERE id = 1") \
        .option("customSchema", "id LONG, message STRING") \
        .load()

    results = df.collect()
    elapsed = (time.time() - start_time) * 1000

    if results:
        print(f"\n[PASS] Read JdbcTest node in {elapsed:.1f}ms")
        df.show(truncate=False)
    else:
        print(f"\n[FAIL] No rows returned - did you run the MERGE Cypher statement first?")
except Exception as e:
    print(f"\n[FAIL] {e}")

---

## Section 3: Unity Catalog JDBC Connection

Create and test the UC JDBC connection. The SafeSpark sandbox wraps the JDBC driver in an isolated JVM.

**Note:** `java_dependencies` only accepts UC Volume paths.

In [ ]:
# Create Unity Catalog JDBC Connection
import time

print(f"Creating UC connection: {UC_CONNECTION_NAME}")

spark.sql(f"DROP CONNECTION IF EXISTS {UC_CONNECTION_NAME}")

create_sql = f"""
CREATE CONNECTION {UC_CONNECTION_NAME} TYPE JDBC
ENVIRONMENT (
  java_dependencies '{JAVA_DEPENDENCIES}'
)
OPTIONS (
  url '{NEO4J_JDBC_URL_SQL}',
  user '{NEO4J_USER}',
  password '{NEO4J_PASSWORD}',
  driver 'org.neo4j.jdbc.Neo4jDriver',
  externalOptionsAllowList 'dbtable,query,partitionColumn,lowerBound,upperBound,numPartitions,fetchSize,customSchema'
)
"""

try:
    start_time = time.time()
    spark.sql(create_sql)
    elapsed = (time.time() - start_time) * 1000
    print(f"\n[PASS] Connection created in {elapsed:.1f}ms")
except Exception as e:
    print(f"\n[FAIL] {e}")

In [ ]:
# Verify connection configuration
spark.sql(f"DESCRIBE CONNECTION {UC_CONNECTION_NAME}").show(truncate=False)

In [ ]:
# UC JDBC - Basic connectivity test
import time

print("TEST: UC JDBC - SELECT 1")

try:
    start_time = time.time()
    df = spark.read.format("jdbc") \
        .option("databricks.connection", UC_CONNECTION_NAME) \
        .option("query", "SELECT 1 AS test") \
        .option("customSchema", "test INT") \
        .load()

    results = df.collect()
    elapsed = (time.time() - start_time) * 1000

    print(f"\n[PASS] UC connection working in {elapsed:.1f}ms")
    df.show()
except Exception as e:
    print(f"\n[FAIL] {e}")

In [ ]:
# UC JDBC - Read the JdbcTest node via SQL through Unity Catalog
import time

print("TEST: UC JDBC - Read JdbcTest node")
print("SQL: SELECT id, message FROM JdbcTest WHERE id = 1")

try:
    start_time = time.time()
    df = spark.read.format("jdbc") \
        .option("databricks.connection", UC_CONNECTION_NAME) \
        .option("query", "SELECT id, message FROM JdbcTest WHERE id = 1") \
        .option("customSchema", "id LONG, message STRING") \
        .load()

    results = df.collect()
    elapsed = (time.time() - start_time) * 1000

    if results:
        print(f"\n[PASS] Read JdbcTest node via UC in {elapsed:.1f}ms")
        df.show(truncate=False)
    else:
        print(f"\n[FAIL] No rows returned")
except Exception as e:
    print(f"\n[FAIL] {e}")